# Escenarios de redistribucion de cultivos

Construye intercambios exclusivamente entre cultivos con modelo validado y esquema historico compatible. Conserva la superficie total de cada municipio-ciclo-modalidad y la superficie nacional de cada cultivo. Por tanto, estima una relocalizacion territorial, no una expansion agricola ni la sustitucion completa de la canasta nacional.

In [1]:
from IPython.display import display
import pandas as pd

from proyecto_agronomia.config import PROCESSED_DATA_DIR
from datasets.inegi import obtener_municipios_inegi
from modeling.cultivos import (
    catalogar_cultivos, cargar_registros_cultivos, seleccionar_principales,
)
from modeling.redistribucion import (
    CLAVES_BLOQUE, construir_matriz_potencial, crear_escenarios,
    cultivos_con_modelo_valido, optimizar_intercambios,
)
from proyecto_agronomia.plots import (
    crear_mapa_redistribucion_cultivos,
    crear_mapa_redistribucion_por_ciclo,
)

2026-11-21 09:02:24.865 | INFO     | proyecto_agronomia.config:<module>:15 - PROJ_ROOT path is: /home/dusty/Documents/Repositories/proyecto_agronomia


/home/dusty/Documents/Repositories/proyecto_agronomia


In [ ]:
RUTA_DATOS = PROCESSED_DATA_DIR / 'agricultura_clima_edafologia_historial_2003_2025.csv'
CANTIDAD_CULTIVOS = 385  # maximo; solo pasan cultivos con cobertura y modelo valido
VERSION_MODELOS = 'v1_principales'
R2_ESPACIAL_MINIMO = 0.50
ANIO_BASE = 2025
ANIOS_CLIMA = (2022, 2025)
MINIMO_ANIOS_HISTORIA = 3
ESCENARIOS = {'conservador': 0.10, 'intermedio': 0.25, 'exploratorio': 0.40}

if not RUTA_DATOS.is_file():
    raise FileNotFoundError(f'No se encontro el dataset historico: {RUTA_DATOS}')
print(f'Dataset para redistribucion: {RUTA_DATOS}')

## Cultivos con modelo y calidad suficiente

In [ ]:
catalogo = catalogar_cultivos(ruta=RUTA_DATOS, unidad='Tonelada')
principales = seleccionar_principales(catalogo, cantidad=CANTIDAD_CULTIVOS)
calidad_modelos = cultivos_con_modelo_valido(
    principales['Nomcultivo'].tolist(), version=VERSION_MODELOS,
    r2_espacial_minimo=R2_ESPACIAL_MINIMO,
)
display(calidad_modelos)
cultivos_elegibles = calidad_modelos.loc[calidad_modelos['elegible'], 'cultivo'].tolist()
assert len(cultivos_elegibles) >= 2, 'No hay suficientes modelos confiables para intercambiar.'
print(f'Cultivos incluidos ({len(cultivos_elegibles)}):', cultivos_elegibles)
print('Cultivos excluidos por calidad:', calidad_modelos.loc[~calidad_modelos['elegible'], 'cultivo'].tolist())

## Matriz municipio-cultivo

Una combinación candidata requiere al menos tres años de historia en el mismo municipio, ciclo y modalidad. El rendimiento esperado es el promedio de predicciones bajo los climas observados entre 2022 y 2025.

In [ ]:
datos_elegibles = cargar_registros_cultivos(
    cultivos_elegibles, ruta=RUTA_DATOS, unidad='Tonelada',
)
matriz_potencial = construir_matriz_potencial(
    datos_elegibles, cultivos=cultivos_elegibles, version=VERSION_MODELOS,
    anio_base=ANIO_BASE, anios_clima=ANIOS_CLIMA,
    minimo_anios_historia=MINIMO_ANIOS_HISTORIA,
)
assert set(matriz_potencial['Nomcultivo']).issubset(set(cultivos_elegibles))
display(pd.DataFrame({
    'combinaciones': [len(matriz_potencial)],
    'municipios': [matriz_potencial['cvegeo'].nunique()],
    'cultivos': [matriz_potencial['Nomcultivo'].nunique()],
    'superficie_base_ha': [matriz_potencial['superficie_base_ha'].sum()],
}))
display(matriz_potencial.head())

## Optimización y escenarios parciales

In [ ]:
superficie_optima, diagnostico = optimizar_intercambios(matriz_potencial)
escenarios = crear_escenarios(matriz_potencial, superficie_optima, ESCENARIOS)
display(pd.Series(diagnostico, name='valor').to_frame())

resumen_escenarios = [{
    'escenario': 'base',
    'produccion_predicha_t': (escenarios['superficie_base_ha'] * escenarios['rendimiento_esperado']).sum(),
    'superficie_reasignada_ha': 0.0,
}]
for nombre in ESCENARIOS:
    resumen_escenarios.append({
        'escenario': nombre,
        'produccion_predicha_t': escenarios[f'produccion_{nombre}_t'].sum(),
        'superficie_reasignada_ha': escenarios[f'cambio_{nombre}_ha'].clip(lower=0).sum(),
    })
resumen_escenarios = pd.DataFrame(resumen_escenarios)
base_t = resumen_escenarios.loc[resumen_escenarios['escenario'].eq('base'), 'produccion_predicha_t'].iloc[0]
resumen_escenarios['ganancia_t'] = resumen_escenarios['produccion_predicha_t'] - base_t
resumen_escenarios['ganancia_pct'] = resumen_escenarios['ganancia_t'] / base_t * 100
display(resumen_escenarios)

## Validar conservación de superficies

In [ ]:
for nombre in ESCENARIOS:
    columna = f'superficie_{nombre}_ha'
    total_bloque = escenarios.groupby(CLAVES_BLOQUE, observed=True)[columna].sum()
    base_bloque = escenarios.groupby(CLAVES_BLOQUE, observed=True)['superficie_base_ha'].sum()
    assert (total_bloque - base_bloque).abs().max() < 1e-4
    claves_cultivo = ['Nomcultivo', 'Nomcicloproductivo', 'Nommodalidad']
    total_cultivo = escenarios.groupby(claves_cultivo, observed=True)[columna].sum()
    base_cultivo = escenarios.groupby(claves_cultivo, observed=True)['superficie_base_ha'].sum()
    assert (total_cultivo - base_cultivo).abs().max() < 1e-4
print('Validacion correcta: se conservan superficies municipales y nacionales por cultivo.')

## Principales intercambios y guardado

In [ ]:
columnas = [
    'cvegeo', 'Nomestado', 'Nommunicipio', 'Nomcicloproductivo',
    'Nommodalidad', 'Nomcultivo', 'superficie_base_ha',
    'superficie_intermedio_ha', 'cambio_intermedio_ha',
    'rendimiento_esperado', 'anios_historia', 'anios_clima',
]
display(escenarios.loc[escenarios['cambio_intermedio_ha'].gt(0), columnas]
        .nlargest(30, 'cambio_intermedio_ha'))

ruta_escenarios = PROCESSED_DATA_DIR / 'redistribucion_cultivos_principales_2025.csv'
ruta_resumen = PROCESSED_DATA_DIR / 'redistribucion_cultivos_resumen_2025.csv'
escenarios.to_csv(ruta_escenarios, index=False)
resumen_escenarios.to_csv(ruta_resumen, index=False)
print(f'Escenarios: {ruta_escenarios}')
print(f'Resumen: {ruta_resumen}')

## Mapas de distribución original y redistribuida

El color representa el cultivo con mayor superficie dentro del conjunto elegible en cada municipio. Ambos mapas utilizan exactamente la misma paleta.

In [ ]:
municipios_geo = obtener_municipios_inegi()
figura_mapa, ejes_mapa, mapa_redistribucion, cambios_dominantes = (
    crear_mapa_redistribucion_cultivos(
        escenarios, municipios_geo, escenario='intermedio',
    )
)
display(cambios_dominantes.loc[cambios_dominantes['cambio_dominante']].head(20))

resumen_ciclos = (
    escenarios.assign(
        produccion_base_t=lambda df: df['superficie_base_ha'] * df['rendimiento_esperado'],
    )
    .groupby('Nomcicloproductivo', as_index=False, observed=True)
    .agg(
        superficie_base_ha=('superficie_base_ha', 'sum'),
        produccion_base_t=('produccion_base_t', 'sum'),
        produccion_intermedia_t=('produccion_intermedio_t', 'sum'),
        superficie_reasignada_ha=('cambio_intermedio_ha', lambda serie: serie.clip(lower=0).sum()),
    )
)
resumen_ciclos['ganancia_t'] = (
    resumen_ciclos['produccion_intermedia_t'] - resumen_ciclos['produccion_base_t']
)
resumen_ciclos['ganancia_pct'] = (
    resumen_ciclos['ganancia_t'] / resumen_ciclos['produccion_base_t'] * 100
)
display(resumen_ciclos)

RUTA_RESUMEN_CICLOS = PROCESSED_DATA_DIR / 'redistribucion_cultivos_por_ciclo_2025.csv'
resumen_ciclos.to_csv(RUTA_RESUMEN_CICLOS, index=False)
figura_ciclos, ejes_ciclos, cambios_por_ciclo = crear_mapa_redistribucion_por_ciclo(
    escenarios, municipios_geo, escenario='intermedio',
)
print(f'Resumen por ciclo: {RUTA_RESUMEN_CICLOS}')

## Interpretación responsable

Los resultados son escenarios de potencial, no instrucciones de reconversión. Mantienen la canasta y superficie observadas, pero todavía no incluyen costos, demanda, disponibilidad de agua, infraestructura, rotaciones ni decisiones de productores.